# Phase 5z Wave 2: Why Neutrinos Have Tiny Masses — Stakeholder Notebook

**Plain-language goal:** Connect three deep facts about the universe — the existence of right-handed neutrinos, why neutrinos have masses 100 billion times smaller than electrons, and a strange algebraic constraint from topology — into a single coherent story, then check it line-by-line in a computer proof.

Three observations the Standard Model doesn't explain:
1. **Why are neutrino masses so tiny?** Atmospheric oscillations require $m_\nu \sim 0.05$ eV (a hundred-millionth of an electron mass).
2. **Where are the right-handed neutrinos?** All other Standard Model fermions come in left-and-right pairs except the neutrino.
3. **Why does the SM-with-three-generations need exactly the matter content it has?** A topological invariant (the $\mathbb{Z}_{16}$ Dai-Freed anomaly) places a hard constraint.

**The seesaw answer (1977):** Add three heavy partner neutrinos $\nu_R$ at scale $M_R \sim 10^{14}$ GeV. Their interaction with the Higgs field generates light-neutrino masses via

$$m_\nu = y^2 \, v^2 / M_R$$

where $v = 246$ GeV is the electroweak vacuum and $y$ is a Yukawa coupling. The 'seesaw' name reflects the inverse relationship: heavy partners up, light masses down.

**The $\mathbb{Z}_{16}$ answer (Garcia-Etxebarria/Wan-Wang):** Each generation's 15 SM Weyl fermions sums to $-1 \pmod{16}$. Three generations give $-3 \pmod{16}$. Adding three $\nu_R$ (each carrying charge $+1$) restores $0 \pmod{16}$ — anomaly-free.

**The Wave-2 contribution:** Both stories live on the same fermionic substrate that hosts the Higgs (Phase 5z Wave 1) and the tetrad (Phase 4y) condensates of our research program. The $\mathbb{Z}_{16}$ saturation and the seesaw are now both machine-verified in Lean 4.

## 1. The Numbers — What Experiments Tell Us

Three-flavor neutrino oscillations (Daya Bay, T2K, NOvA, KamLAND, Super-K, IceCube) determine three mixing angles, two mass-squared splittings, and a CP-violating phase. The 2024 NuFit-6.0 global fit gives:

In [1]:
import os, sys
_HERE = os.getcwd()
_PROJ = _HERE if os.path.basename(_HERE) == 'SK_EFT_Hawking' else os.path.dirname(_HERE)
if _PROJ not in sys.path:
    sys.path.insert(0, _PROJ)

from src.core.constants import EW_PARAMS, MAJORANA_PARAMS
from src.core.formulas import seesaw_neutrino_mass, seesaw_m_r_from_observed

print('Measured (NuFit-6.0, normal ordering):')
print(f"  Solar mass-splitting:       Δm²_21 = {MAJORANA_PARAMS['DELTA_M_SQ_21_EV2']:.2e} eV²")
print(f"  Atmospheric mass-splitting: |Δm²_31| = {MAJORANA_PARAMS['DELTA_M_SQ_31_EV2']:.2e} eV²")
print(f"  Atmospheric mixing angle:   θ_23 = {MAJORANA_PARAMS['THETA_23_DEG']:.1f}° (near maximal)")
print(f"  Solar mixing angle:         θ_12 = {MAJORANA_PARAMS['THETA_12_DEG']:.1f}°")
print(f"  Reactor mixing angle:       θ_13 = {MAJORANA_PARAMS['THETA_13_DEG']:.1f}° (small)")
print()
print(f"Heaviest light-ν mass (assuming m_lightest → 0):")
print(f"  m_3 ≈ √|Δm²_31| ≈ {MAJORANA_PARAMS['M_NU_HEAVIEST_EV']*1000:.1f} meV")

Measured (NuFit-6.0, normal ordering):
  Solar mass-splitting:       Δm²_21 = 7.42e-05 eV²
  Atmospheric mass-splitting: |Δm²_31| = 2.51e-03 eV²
  Atmospheric mixing angle:   θ_23 = 49.1° (near maximal)
  Solar mixing angle:         θ_12 = 33.4°
  Reactor mixing angle:       θ_13 = 8.5° (small)

Heaviest light-ν mass (assuming m_lightest → 0):
  m_3 ≈ √|Δm²_31| ≈ 50.1 meV


## 2. The Seesaw — Heavy Means Light

The seesaw says: if the right-handed Majorana mass $M_R$ is heavy, the light-neutrino mass $m_\nu$ is small. Specifically, $m_\nu = y^2 v^2 / M_R$. Quick examples:

In [2]:
v = EW_PARAMS['V_EW_GEV']
print('Reproducing m_3 ≈ 50 meV via seesaw:')
for y, label in [(1e-3, 'small Yukawa'), (1.0, 'O(1) Yukawa, top-like')]:
    M_R = seesaw_m_r_from_observed(y, v, MAJORANA_PARAMS['M_NU_HEAVIEST_EV'] * 1e-9)
    print(f'  y = {y:.0e} ({label:<25}) → M_R = {M_R:.2e} GeV')

print()
print('The seesaw signature: factor-of-10 increase in M_R → factor-of-10 decrease in m_ν.')
for M_R_gev in [1e13, 1e14, 1e15]:
    m_nu_gev = seesaw_neutrino_mass(1.0, v, M_R_gev)
    print(f'  M_R = {M_R_gev:.0e} GeV → m_ν = {m_nu_gev*1e9*1000:.2f} meV')

Reproducing m_3 ≈ 50 meV via seesaw:
  y = 1e-03 (small Yukawa             ) → M_R = 1.21e+09 GeV
  y = 1e+00 (O(1) Yukawa, top-like    ) → M_R = 1.21e+15 GeV

The seesaw signature: factor-of-10 increase in M_R → factor-of-10 decrease in m_ν.
  M_R = 1e+13 GeV → m_ν = 6062.41 meV
  M_R = 1e+14 GeV → m_ν = 606.24 meV
  M_R = 1e+15 GeV → m_ν = 60.62 meV


## 3. The $\mathbb{Z}_{16}$ Topology Constraint — Counting in Modular Arithmetic

There is a deep mathematical structure (the Dai-Freed anomaly) that places a hard constraint on the matter content of any consistent quantum field theory. For the SM, the constraint is: **the total number of Weyl fermions must be a multiple of 16**.

Without right-handed neutrinos:
- One generation = 15 Weyl fermions.
- Three generations = 45 Weyl fermions.
- $45 \bmod 16 = 13 = -3 \pmod{16}$. **Three units short.**

With three right-handed neutrinos:
- $45 + 3 = 48$.
- $48 \bmod 16 = 0$. **Exactly anomaly-free.**

This is the project's `Z16AnomalyComputation.three_nu_R_cancel_three_gen` theorem. Wave 2 packages this as the load-bearing bridge `MajoranaRung.majorana_rung_compatible_with_hidden_singlet`.

In [3]:
from src.core.formulas import majorana_rung_z16_compatibility_index

print('Three generations of SM-without-ν_R need a +3 mod 16 hidden sector to cancel anomalies.')
print('Three fundamental ν_R Weyl fermions, each carrying ℤ_16 charge +1, do exactly that:')
print()
for n_nu_r in [0, 1, 2, 3]:
    contribution = majorana_rung_z16_compatibility_index(n_nu_r)
    total = (45 + n_nu_r) % 16
    status = ' ANOMALY-FREE' if total == 0 else ''
    print(f'  {n_nu_r} ν_R contribute +{contribution} mod 16. SM total: {total} mod 16.{status}')

print()
print('This is the Embedding-III "Hybrid" choice (per the deep-research scoping report):')
print('  - ν_R is a fundamental field in our Lean formalization.')
print('  - Its UV interpretation is a bound state of the underlying ADW substrate.')
print('  - The bridge to ADW substrate is a tracked hypothesis (open in primary literature).')

Three generations of SM-without-ν_R need a +3 mod 16 hidden sector to cancel anomalies.
Three fundamental ν_R Weyl fermions, each carrying ℤ_16 charge +1, do exactly that:

  0 ν_R contribute +0 mod 16. SM total: 13 mod 16.
  1 ν_R contribute +1 mod 16. SM total: 14 mod 16.
  2 ν_R contribute +2 mod 16. SM total: 15 mod 16.
  3 ν_R contribute +3 mod 16. SM total: 0 mod 16. ANOMALY-FREE

This is the Embedding-III "Hybrid" choice (per the deep-research scoping report):
  - ν_R is a fundamental field in our Lean formalization.
  - Its UV interpretation is a bound state of the underlying ADW substrate.
  - The bridge to ADW substrate is a tracked hypothesis (open in primary literature).


## 4. The Seesaw $(y, M_R)$ Plane

Visualize the seesaw: each diagonal contour fixes the light-neutrino mass at one of the NuFit-6.0 anchors. The shaded band is the natural seesaw region; canonical ν_R live in the upper-right corner (heavy + Yukawa-of-order-unity).

In [4]:
from src.core.visualizations import fig_seesaw_y_m_r_band
# viz-ref: fig_seesaw_y_m_r_band
fig_seesaw_y_m_r_band()

## 5. Looking for $0\nu\beta\beta$ — The Cleanest Test

If neutrinos are Majorana particles (their own antiparticles), they enable a process called **neutrinoless double-beta decay** ($0\nu\beta\beta$): a nucleus emits two electrons but no neutrinos, violating lepton number by 2 units. The rate is set by an effective Majorana mass $m_{\beta\beta}$ that depends on the absolute mass scale and the Majorana phases.

**The current bound (KamLAND-Zen 800, 2024):** $m_{\beta\beta} < 28$–$122$ meV (90% CL; uncertainty from nuclear matrix elements).

**The next-generation reach (LEGEND-1000, 2030s):** $m_{\beta\beta} \approx 9$–$21$ meV, covering all of the inverted-ordering parameter space.

Both bounds are embedding-agnostic — they don't depend on the substrate physics — but they are decisive: if no $0\nu\beta\beta$ signal appears at LEGEND-1000, neutrinos either are not Majorana or have normal-ordered masses with $m_{\rm lightest}$ small.

In [5]:
from src.core.visualizations import fig_m_beta_beta_vs_m_lightest
# viz-ref: fig_m_beta_beta_vs_m_lightest
fig_m_beta_beta_vs_m_lightest()

## 6. What Wave 2 Adds — Honestly

**What's machine-verified now:**
- Three fundamental ν_R Weyl fermions saturate $\mathbb{Z}_{16}$ anomaly cancellation. *No new axiom required* — this follows from existing project theorems.
- The Type-I seesaw mass relation, including the strict-monotonicity-in-$M_R$ signature.
- The PMNS mixing matrix as a structural type (3×3 unitary on $\mathbb{C}$).
- A falsifiability predicate testing whether predicted $m_\nu$ matches observation within tolerance.

**What's NOT yet derived (tracked as open hypotheses):**
- $M_R = c \cdot \Lambda_{\rm ADW}$ for some coefficient $c \in (0, 1]$. The substrate-bridge identification is open in primary literature.
- Mixing angles from substrate overlaps (especially $\theta_{23} \approx \pi/4$ from a $\mu$-$\tau$ symmetry).
- Majorana phases from composite-operator structure.
- $m_{\beta\beta}$ tied to a single condensate scale.
- ν_R-as-substrate-bound-state IR-equivalence.

**Why this matters:** The bridge from $\mathbb{Z}_{16}$ anomaly cancellation to the sterile-neutrino seesaw is now formally verified. The five 'open' flags are honestly surfaced rather than papered over. This is the difference between a publishable formal-verification result and a hand-wave.